In [ ]:
# define Meta data

# rec_loops = 3



Model reference: [Less is More: Recursive Reasoning with Tiny Networks](https://arxiv.org/abs/2510.04871). The model below is a simplified MLP version inspired by TRM.


## 4-digit addition, seed 100

Halt target: `softmean`. Budget: 100 epochs per main run.

The five training calls below compare halt thresholds and outer supervision-step limits (`n_sup`).


In [ ]:
import torch
from torch import nn

## Model

The same MLP updates `z` (latent state) and `y` (answer state). The token head and halt head both read `y`. A sample stops when `halt_logit > HALT_THRESHOLD`, or when it has used all `n_sup` supervision steps.


Each forward pass has `T-1` update blocks under `no_grad`, followed by one block with gradients. Keep this distinction in mind for the one-step runs: `n_sup=1` still includes the inner recursion.


In [ ]:
class TinyRModel(nn.Module):
  def __init__(self, hidden_size, vocab_size, inp_seq_len, out_seq_len):
    super().__init__()
    self.net = nn.Sequential(nn.Linear(hidden_size, hidden_size),
                              nn.ReLU(),
                              nn.Linear(hidden_size, hidden_size))

    self.embedding = nn.Embedding(vocab_size, hidden_size)
    self.input_proj = nn.Linear(inp_seq_len * hidden_size, hidden_size)


    self.out = nn.Linear(hidden_size, out_seq_len * vocab_size)
    self.halt = nn.Linear(hidden_size, 1)



  def latent(self, x, y, z, n=6):
      # The latent updates use the input as well as both states.
      for i in range(n):
          z = self.net(x + y + z)

      # The answer update uses the same network, without adding x again.
      y = self.net(y + z)

      return y, z



  def forward(self, x, y, z, T, n=6):

    x = self.embedding(x)          # (B, L, H)
    x = x.view(x.size(0), -1)
    x = self.input_proj(x)                 # (B, H)

    with torch.no_grad():
      for j in range(T-1):
        y, z= self.latent(x,y,z,n)

    y, z = self.latent(x, y, z, n)

    f_out = self.out(y)

    halt= self.halt(y)

    return f_out, halt, y, z

Old majority-task helper, left commented out. The addition runs do not use it.


In [ ]:
# Simple dataset

# import torch
# import random
# from torch.utils.data import Dataset, DataLoader, random_split

# # ── config ───────────────────────────────────────────────────────────────
# hidden_size = 64
# batch_size  = 32
# device      = 'cuda' if torch.cuda.is_available() else 'cpu'
# vocab_size  = 2    # only real tokens: 0 and 1
# num_classes = 2    # label 0 or 1
# seq_len     = 15   # fixed length — no padding needed

# # ── dataset ──────────────────────────────────────────────────────────────
# class MajorityDataset(Dataset):
#     """
#     Fixed-length binary sequences, no padding, no masking.
#     Label = 1 if 1s are majority, 0 if 0s are majority.
#     Ties skipped to avoid ambiguous labels.
#     """
#     def __init__(self, num_samples=10000, seq_len=seq_len):
#         self.samples = self._generate(num_samples, seq_len)

#     def _generate(self, n, seq_len):
#         samples = []
#         while len(samples) < n:
#             seq   = [random.randint(0, 1) for _ in range(seq_len)]
#             ones  = sum(seq)
#             zeros = seq_len - ones
#             if ones == zeros:       # skip ties
#                 continue
#             label = 1 if ones > zeros else 0
#             samples.append((seq, label))
#         return samples

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         seq, label = self.samples[idx]
#         return {
#             "input_ids": torch.tensor(seq,   dtype=torch.long),
#             "labels":    torch.tensor(label, dtype=torch.long),
#         }

# # ── dataloaders ──────────────────────────────────────────────────────────
# dataset  = MajorityDataset(num_samples=10000)
# val_size = 1000
# train_set, val_set = random_split(dataset, [len(dataset) - val_size, val_size])

# train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
# val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)

## Data

50,000 pairs of 4-digit operands, generated with seed 100. Split: 45,000 training examples and 5,000 validation examples.


In [ ]:
import torch
import random
from torch.utils.data import Dataset, DataLoader, random_split

random.seed(100)
torch.manual_seed(100)

def generate_pair(num_digits):
    a = random.randint(10**(num_digits-1), 10**num_digits - 1)
    b = random.randint(10**(num_digits-1), 10**num_digits - 1)
    return str(a), str(b)


operands_a, operands_b = [], []

for sample_idx in range(50000):
    num_a, num_b = generate_pair(num_digits=4)

    operands_a.append(num_a)
    operands_b.append(num_b)

In [ ]:
dataset = []

for num_a, num_b in zip(operands_a, operands_b):

    input_sequence = list(num_a) + ['+'] + list(num_b)

    rev_a = list(reversed(num_a))
    rev_b = list(reversed(num_b))

    num_digits = len(num_a)

    result_digits = []
    carry_sequence = []

    carry = 0
    current_chain_length = 0
    max_carry_chain = 0

    for digit_a, digit_b in zip(rev_a, rev_b):

        digit_sum = int(digit_a) + int(digit_b) + carry

        output_digit = digit_sum % 10
        new_carry = digit_sum // 10

        result_digits.append(output_digit)
        carry_sequence.append(new_carry)

        if new_carry == 1:
            current_chain_length += 1
            max_carry_chain = max(max_carry_chain, current_chain_length)
        else:
            current_chain_length = 0

        carry = new_carry

    if carry > 0:
        result_digits.append(carry)

    final_output = list(reversed(result_digits))

    data = {
        'input': input_sequence,
        'output': final_output,
        'num_digits': num_digits,
        'carry_sequence': carry_sequence,
        'max_carry_chain': max_carry_chain
    }

    dataset.append(data)

print(dataset[0], len(dataset))

{'input': ['3', '3', '8', '6', '+', '8', '5', '2', '8'], 'output': [1, 1, 9, 1, 4], 'num_digits': 4, 'carry_sequence': [1, 1, 0, 1], 'max_carry_chain': 2} 50000


In [ ]:
import torch
from torch.utils.data import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

PAD_TOKEN = 11
PLUS_TOKEN = 10

inp_seq_len = 9
out_seq_len = 5


class AdditionDataset(Dataset):
    def __init__(self, dataset_list, max_input_len=inp_seq_len, max_output_len=out_seq_len):
        self.data = dataset_list
        self.max_input_len = max_input_len
        self.max_output_len = max_output_len

    def pad(self, seq, max_len, pad_value=PAD_TOKEN):
        return seq + [pad_value] * (max_len - len(seq))

    def encode_input(self, input_seq):
        return [
            int(x) if x != '+' else PLUS_TOKEN
            for x in input_seq
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        input_ids = self.encode_input(sample['input'])
        labels    = sample['output']

        input_ids = self.pad(input_ids, self.max_input_len)
        labels    = self.pad(labels, self.max_output_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "carry": torch.tensor(sample['max_carry_chain'], dtype=torch.long)
        }



hidden_size = 256

batch_size  = 32

vocab_size = 12

dataset = AdditionDataset(dataset)

val_size = 5000
train_size = len(dataset) - val_size

torch.manual_seed(100)
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size)

## Loss

For `softmean`, the halt target is the fraction of non-padding answer tokens predicted correctly. The halt head uses binary cross-entropy with logits, with weight `c=0.1`.

Padding needs care here. Padded tokens contribute zero prediction loss, but the main loop still counts them in the active-token denominator. That affects the loss used for training as well as the printed CE.


In [ ]:
# def Q_LOSS(q_halt, f_out, y_true):
#     print(f_out.argmax(dim=-1), y_true)
#     target = f_out.argmax(dim=-1) == y_true
#     print(target)
#     return F.binary_cross_entropy_with_logits(q_halt, target)

In [ ]:
# def model_training_and_validation_with_mask(c, T, n_sup):
#     model = TinyRModel(hidden_size, vocab_size, inp_seq_len=12, out_seq_len=6).to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#     print('PAD_TOKEN: ', PAD_TOKEN)
#     criterion = nn.CrossEntropyLoss(reduction='none', ignore_index=PAD_TOKEN)

#     running_loss = 0
#     running_q = 0

#     print('Training Logs ------ \n')
#     model.train()
#     for step, batch in enumerate(train_loader):
#         input_ids = batch['input_ids'].to(device)
#         labels = batch['labels'].to(device)
#         batch_size_curr = input_ids.size(0)
#         seq_len = labels.size(1)

#         active_mask = torch.ones(batch_size_curr, dtype=torch.bool, device=device)

#         x = input_ids
#         y = torch.zeros(batch_size_curr, hidden_size, device=device)
#         z = torch.zeros(batch_size_curr, hidden_size, device=device)

#         step_loss = step_q = 0
#         overall_loss = torch.tensor(0.0, device=device)

#         for i in range(n_sup):
#             y_new, z_new, halt, out = model(x, y, z, halt, out)


#             y = torch.where(active_mask.unsqueeze(1), y_new.detach(), y)
#             z = torch.where(active_mask.unsqueeze(1), z_new.detach(), y)

#             loss_per_token = crossntropy(f_out.view(-1, vocab_size), labels.view(-1))




In [ ]:
import torch.nn.functional as F

def Q_LOSS(q_halt, f_out, y_true, pad_token=PAD_TOKEN):
    preds = f_out.argmax(dim=-1)                     # (B, L)
    correct = (preds == y_true).float()              # (B, L)
    non_pad = (y_true != pad_token).float()          # (B, L)
    target = (correct * non_pad).sum(dim=-1, keepdim=True) / (non_pad.sum(dim=-1, keepdim=True) + 1e-8)
    return F.binary_cross_entropy_with_logits(q_halt, target, reduction='none')

In [ ]:
# AI-generated diagnostic code (Claude), guided by the experiment author.


from torch.utils.data import Subset

torch.manual_seed(100)

def diagnostic_per_sup_step(c=0.1, T=3, n_sup=16, num_epochs=100):
    model = TinyRModel(hidden_size, vocab_size, inp_seq_len = inp_seq_len, out_seq_len = out_seq_len).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion_noreduce = nn.CrossEntropyLoss(reduction='none', ignore_index=PAD_TOKEN)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

    diag_subset = Subset(train_set, range(max(10000, len(train_set))))
    diag_loader = DataLoader(diag_subset, batch_size=64, shuffle=True)

    model.train()

    for epoch in range(num_epochs):
        print(f'processing epoch {epoch} \n')
        for batch in diag_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            batch_size_curr = input_ids.size(0)
            seq_len = labels.size(1)

            x = input_ids
            y = torch.zeros(batch_size_curr, hidden_size, device=device)
            z = torch.zeros(batch_size_curr, hidden_size, device=device)

            total_loss = torch.tensor(0.0, device=device)

            for i in range(n_sup):
                f_out, halt, y_new, z_new = model(x, y, z, T, n=6)
                y = y_new.detach()
                z = z_new.detach()

                loss = criterion(f_out.view(-1, vocab_size), labels.view(-1))

                # Use the same halt target as the main training run.
                seq_len = labels.size(1)
                q_loss = Q_LOSS(halt, f_out.view(-1, seq_len, vocab_size), labels)

                total_loss += (loss + c * q_loss.mean())

            total_loss /= n_sup

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

        # Compare supervision steps on one training batch every ten epochs.
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                test_batch = next(iter(diag_loader))
                input_ids = test_batch['input_ids'].to(device)
                labels = test_batch['labels'].to(device)
                batch_size_curr = input_ids.size(0)
                seq_len = labels.size(1)

                x = input_ids
                y_state = torch.zeros(batch_size_curr, hidden_size, device=device)
                z_state = torch.zeros(batch_size_curr, hidden_size, device=device)

                print(f"Epoch {epoch:3d} | CE per supervision step (same batch):")

                prev_ce = None
                for i in range(n_sup):
                    f_out, halt, y_new, z_new = model(x, y_state, z_state, T, n=6)
                    y_state = y_new.detach()
                    z_state = z_new.detach()

                    print(f"sup_step_{i+1} | mean={halt.mean().item():.3f} "
                          f"std={halt.std().item():.3f} "
                          f"median={halt.median().item():.3f} "
                          f"%>0={(halt > 0).float().mean().item():.1%} "
                          f"first5={halt[:5].squeeze().tolist()}")

                    loss_per_token = criterion_noreduce(f_out.view(-1, vocab_size), labels.view(-1))
                    loss_grid = loss_per_token.view(batch_size_curr, seq_len)
                    non_pad = (labels != PAD_TOKEN)
                    # Each sample is averaged over its non-padding tokens.
                    ce_per_sample = (loss_grid * non_pad.float()).sum(dim=1) / non_pad.float().sum(dim=1).clamp(min=1)
                    avg_ce = ce_per_sample.mean().item()

                    preds = f_out.view(-1, seq_len, vocab_size).argmax(dim=-1)
                    correct = (preds == labels) & non_pad
                    token_acc = correct.sum().item() / non_pad.sum().item()

                    # Padding does not count against whole-answer correctness.
                    per_sample_correct = ((preds == labels) | ~non_pad).all(dim=-1).float()
                    exact = per_sample_correct.mean().item()

                    delta = ""
                    if prev_ce is not None:
                        diff = prev_ce - avg_ce
                        delta = f"| Delta: {diff:+.4f} {'IMPROVED' if diff > 0.01 else 'FLAT' if diff > -0.01 else 'WORSE'}"
                    prev_ce = avg_ce

                    pred_strings = [tuple(p.cpu().tolist()) for p in preds]
                    unique_preds = len(set(pred_strings))

                    print(f"  Sup {i:2d} | CE: {avg_ce:.4f} | TokAcc: {token_acc:.4f} | Exact: {exact:.4f} | Unique: {unique_preds}/{batch_size_curr} {delta}")

                print()

                # Show the last step with padding removed.
                if epoch % 50 == 0:
                    print(f"--- Sample Predictions at Epoch {epoch}, Last Sup Step ---")
                    for j in range(min(10, batch_size_curr)):
                        mask = non_pad[j]
                        pred_real = preds[j][mask].cpu().tolist()
                        true_real = labels[j][mask].cpu().tolist()
                        inp = input_ids[j].cpu().tolist()
                        inp_str = ''.join([str(t) if t < 10 else '+' if t == 10 else '' for t in inp])
                        match = "MATCH" if pred_real == true_real else "WRONG"
                        print(f"  {inp_str} | Pred: {pred_real} True: {true_real} {match}")
                    print()

            model.train()

    # Compare all steps once more after training.
    print("=" * 70)
    print("FINAL BREAKDOWN: All Supervision Steps (Last Epoch)")
    print("=" * 70)

    model.eval()
    with torch.no_grad():
        test_batch = next(iter(diag_loader))
        input_ids = test_batch['input_ids'].to(device)
        labels = test_batch['labels'].to(device)
        batch_size_curr = input_ids.size(0)
        seq_len = labels.size(1)

        x = input_ids
        y_state = torch.zeros(batch_size_curr, hidden_size, device=device)
        z_state = torch.zeros(batch_size_curr, hidden_size, device=device)

        non_pad = (labels != PAD_TOKEN)

        for i in range(n_sup):
            f_out, halt, y_new, z_new = model(x, y_state, z_state, T, n=6)
            y_state = y_new.detach()
            z_state = z_new.detach()

            loss_per_token = criterion_noreduce(f_out.view(-1, vocab_size), labels.view(-1))
            loss_grid = loss_per_token.view(batch_size_curr, seq_len)
            ce_per_sample = (loss_grid * non_pad.float()).sum(dim=1) / non_pad.float().sum(dim=1).clamp(min=1)
            avg_ce = ce_per_sample.mean().item()

            preds = f_out.view(-1, seq_len, vocab_size).argmax(dim=-1)
            correct = (preds == labels) & non_pad
            token_acc = correct.sum().item() / non_pad.sum().item()

            per_sample_correct = ((preds == labels) | ~non_pad).all(dim=-1).float()
            exact = per_sample_correct.mean().item()

            pred_strings = [tuple(p.cpu().tolist()) for p in preds]
            unique_preds = len(set(pred_strings))

            q_mean = halt.mean().item()

            print(f"  Sup {i:2d} | CE: {avg_ce:.4f} | TokAcc: {token_acc:.4f} | Exact: {exact:.4f} | Unique: {unique_preds}/{batch_size_curr} | Q: {q_mean:.4f}")

        print(f"\n--- Final Sample Predictions ---")
        for j in range(min(10, batch_size_curr)):
            mask = non_pad[j]
            pred_real = preds[j][mask].cpu().tolist()
            true_real = labels[j][mask].cpu().tolist()
            inp = input_ids[j].cpu().tolist()
            inp_str = ''.join([str(t) if t < 10 else '+' if t == 10 else '' for t in inp])
            match = "MATCH" if pred_real == true_real else "WRONG"
            print(f"  {inp_str} | Pred: {pred_real} True: {true_real} {match}")

    return model

model = diagnostic_per_sup_step(c=0.1, T=3, n_sup=16, num_epochs=100)

processing epoch 0 

Epoch   0 | CE per supervision step (same batch):
sup_step_1 | mean=-0.554 std=0.307 median=-0.407 %>0=0.0% first5=[-0.9460815191268921, -0.31768324971199036, -0.9671576023101807, -0.29262450337409973, -0.4071010947227478]
  Sup  0 | CE: 1.7540 | TokAcc: 0.3333 | Exact: 0.0000 | Unique: 36/64 
sup_step_2 | mean=-0.613 std=0.334 median=-0.466 %>0=0.0% first5=[-0.964099645614624, -0.2586132884025574, -1.0440235137939453, -0.2901744842529297, -0.40028491616249084]
  Sup  1 | CE: 1.6587 | TokAcc: 0.3704 | Exact: 0.0000 | Unique: 38/64 | Delta: +0.0953 IMPROVED
sup_step_3 | mean=-0.633 std=0.336 median=-0.512 %>0=0.0% first5=[-0.9861810207366943, -0.3226962685585022, -1.0620182752609253, -0.23585209250450134, -0.3753618896007538]
  Sup  2 | CE: 1.6760 | TokAcc: 0.3603 | Exact: 0.0000 | Unique: 33/64 | Delta: -0.0173 WORSE
sup_step_4 | mean=-0.635 std=0.338 median=-0.509 %>0=0.0% first5=[-0.9971588850021362, -0.32520756125450134, -1.0653433799743652, -0.22331717610359192

In [ ]:
# Notes from development
#
# Early debugging notes mixed preliminary runs with working hypotheses. The
# checked comparisons are in docs/technical_report.md and docs/seeded_results.md.
# These are the implementation details that matter when reading the outputs:
#
# - Avg Steps is the mean number of loop iterations per batch. The separate
#   halt-step distribution records individual samples. A sample recorded at
#   n_sup may have crossed the threshold there or simply reached the limit.
# - A step-1 exit does not imply a correct answer. The halt head learns the
#   chosen target: token accuracy, confidence, or whole-answer correctness.
# - Padding is excluded from halt targets and accuracy. The main prediction
#   loss still divides by all active token positions, including padding.
# - Q_LOSS returns one loss per sample, so the active mask can exclude samples
#   that have already halted before the loss is averaged.
# - For a fixed target t, the BCE derivative with respect to logit h is
#   sigmoid(h) - t. Its isolated optimum is logit(t) for 0 < t < 1; shared
#   weights and changing targets make this an interpretation, not a guarantee.
# - The rule h > threshold compares the learned score sigmoid(h) with
#   sigmoid(threshold). It does not compare the true target directly.
#   Raising the threshold need not lead to complete collapse within a run.
# - Binary targets of zero still produce a BCE gradient. They teach the halt
#   head to continue; they do not remove its learning signal.
# - If every sample halts at step 1, later supervision steps are skipped and
#   add no direct loss. The shared network still trains on step 1.
# - One supervision step still performs inner recursion. Similar halt logits
#   at successive steps alone do not establish that the hidden state is fixed.
#
# The saved runs compare three targets with one halt head. They do not test
# warmup, a second head, or tasks that need more dependent reasoning steps.


In [ ]:
import torch
import torch.nn.functional as F
from collections import Counter

# Training and validation

def model_training_and_validation_with_mask(c, T, n_sup, HALT_THRESHOLD):

  torch.manual_seed(100)

  model = TinyRModel(hidden_size, vocab_size, inp_seq_len=inp_seq_len, out_seq_len=out_seq_len).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

  print('PAD_TOKEN: ', PAD_TOKEN)
  criterion = nn.CrossEntropyLoss(reduction='none', ignore_index=PAD_TOKEN)

  epochs = 100


  print('Training Logs ------ \n')
  for epoch in range(epochs):
        model.train()

        running_loss = 0
        running_q = 0
        running_n_taken = 0

        all_halt_steps = []
        all_halt_logits = []

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            batch_size_curr = input_ids.size(0)
            seq_len = labels.size(1)

            active_mask = torch.ones(batch_size_curr, dtype=torch.bool, device=device)

            x = input_ids
            y = torch.zeros(batch_size_curr, hidden_size, device=device)
            z = torch.zeros(batch_size_curr, hidden_size, device=device)

            step_loss = step_q = 0
            overall_loss = torch.tensor(0.0, device=device)

            # Samples that never cross the threshold keep the step limit.
            halt_step = torch.full((batch_size_curr,), n_sup, dtype=torch.long, device=device)


            for i in range(n_sup):
                f_out, halt, y_new, z_new = model(x, y, z, T, n=6)

                all_halt_logits.append(halt.squeeze(-1).detach().cpu())

                # Detach between outer steps and keep halted samples' states fixed.
                y = torch.where(active_mask.unsqueeze(1), y_new.detach(), y)
                z = torch.where(active_mask.unsqueeze(1), z_new.detach(), z)

                seq_len = labels.size(1)  # Padded answer length for this dataset.

                loss_per_token = criterion(f_out.view(-1, vocab_size), labels.view(-1))

                q_loss_per_sample = Q_LOSS(halt, f_out.view(-1, seq_len, vocab_size), labels)

                # This mask excludes halted samples but still counts padding positions.


                token_mask = active_mask.unsqueeze(1).expand(-1, seq_len).reshape(-1).float()
                sample_mask = active_mask.float()

                num_active_tokens = token_mask.sum() + 1e-8
                num_active_samples = sample_mask.sum() + 1e-8

                # Padding contributes zero CE, but remains in this denominator.
                current_step_loss = (loss_per_token * token_mask).sum() / num_active_tokens
                current_step_q = (q_loss_per_sample.squeeze(-1) * sample_mask).sum() / num_active_samples

                overall_loss += (current_step_loss + c * current_step_q)

                step_loss += current_step_loss.item()
                step_q += current_step_q.item()

                halt_decision = (halt.squeeze(-1) > HALT_THRESHOLD) & active_mask

                halt_step[halt_decision] = i + 1

                active_mask = active_mask & (~halt_decision)

                if active_mask.sum() == 0:
                  break

            all_halt_steps.append(halt_step.cpu())

            n_taken = i + 1
            # Average the accumulated loss over the steps this batch reached.
            overall_loss /= n_taken

            # Avg Steps tracks batch loop length; halt_step tracks individual exits.
            running_n_taken += n_taken

            optimizer.zero_grad()
            overall_loss.backward()
            optimizer.step()

            running_loss += step_loss / n_taken
            running_q += step_q / n_taken



        avg_loss = running_loss / len(train_loader)
        avg_q_loss = running_q / len(train_loader)

        avg_steps = running_n_taken / len(train_loader)

        print(f'Training processed epoch {epoch}')
        if epoch % 10 == 0:
            print(f" Training epoch {epoch} | CE: {avg_loss:.4f} | Q-Loss: {avg_q_loss:.4f} | Avg Steps: {avg_steps}")

            all_halt_steps = torch.cat(all_halt_steps)
            all_halt_logits = torch.cat(all_halt_logits)

            mean_h = all_halt_steps.float().mean().item()
            median_h = all_halt_steps.float().median().item()
            frac_1 = (all_halt_steps == 1).float().mean().item()
            frac_max = (all_halt_steps == n_sup).float().mean().item()
            dist = dict(Counter(all_halt_steps.tolist()))

            print(f"halt logit: mean={all_halt_logits.mean().item():.3f} "
                  f"std={all_halt_logits.std().item():.3f} "
                  f"max={all_halt_logits.max().item():.3f} "
                  f"%>0={(all_halt_logits > 0).float().mean().item():.1%} "
                  f"%>1.5={(all_halt_logits > 1.5).float().mean().item():.1%}")

            print(f"halt steps: mean={mean_h:.2f} median={median_h:.0f} "
                  f"%step1={frac_1:.1%} %never={frac_max:.1%} dist={dist}")

        running_loss = running_q = 0


        if epoch % 10 == 0:
            print('\nValidation Logs ------ \n')
            model.eval()

            def validation(use_mask):
              correct = 0
              total = 0

              step_exact     = torch.zeros(n_sup, 5)
              step_count     = torch.zeros(n_sup, 5)

              step_tok_right = torch.zeros(n_sup)
              step_tok_total = torch.zeros(n_sup)

              all_halt_steps_validation = []
              all_exact_validation = []

              with torch.no_grad():
                  for val_step, batch in enumerate(val_loader):
                      input_ids = batch['input_ids'].to(device)
                      labels = batch['labels'].to(device)
                      batch_size_curr = input_ids.size(0)

                      batch_size_curr = input_ids.size(0)
                      seq_len = labels.size(1)

                      active_mask = torch.ones(batch_size_curr, dtype=torch.bool, device=device)

                      # Samples that never cross the threshold keep the step limit.
                      val_halt_step = torch.full((batch_size_curr,), n_sup, dtype=torch.long, device=device)

                      x = input_ids
                      y = torch.zeros(input_ids.size(0), hidden_size, device=device)
                      z = torch.zeros(input_ids.size(0), hidden_size, device=device)

                      final_logits = torch.zeros(batch_size_curr, out_seq_len * vocab_size, device=device)

                      for i in range(n_sup):
                          f_out_val, val_halt, y_new, z_new = model(x, y, z, T, n=6)
                          y = torch.where(active_mask.unsqueeze(1), y_new.detach(), y)
                          z = torch.where(active_mask.unsqueeze(1), z_new.detach(), z)

                          if use_mask and val_step == 0:
                              print(f"sup_step_{i+1} | mean={val_halt.mean().item():.3f} "
                                    f"std={val_halt.std().item():.3f} "
                                    f"median={val_halt.median().item():.3f} "
                                    f"%>0={(val_halt > 0).float().mean().item():.1%} "
                                    f"%>1.5={(val_halt > 1.5).float().mean().item():.1%} "
                                    f"first5={val_halt[:5].squeeze().tolist()}")


                          if use_mask:
                            halt_decision = (val_halt.squeeze(-1) > HALT_THRESHOLD) & active_mask
                            val_halt_step[halt_decision] = i + 1
                            active_mask = active_mask & (~halt_decision)
                            final_logits[halt_decision] = f_out_val[halt_decision]

                            if active_mask.sum() == 0:
                                break

                          else:
                            preds_i = f_out_val.view(-1, seq_len, vocab_size).argmax(dim=-1)
                            np_i = (labels != PAD_TOKEN)
                            ex = ((preds_i == labels) | ~np_i).all(dim=-1)
                            carry = batch['carry'].to(device)

                            for k in range(5):
                                m = (carry == k)
                                step_exact[i, k] += (ex & m).sum().item()
                                step_count[i, k] += m.sum().item()

                            step_tok_right[i] += ((preds_i == labels) & np_i).sum().item()
                            step_tok_total[i] += np_i.sum().item()


                      if use_mask:
                        all_halt_steps_validation.append(val_halt_step.cpu())
                        final_logits[active_mask] = f_out_val[active_mask]

                        preds = final_logits.view(-1, seq_len, vocab_size).argmax(dim=-1)
                        non_pad = (labels != PAD_TOKEN)
                        exact = ((preds == labels) | ~non_pad).all(dim=-1)
                        correct += exact.sum().item()

                        all_exact_validation.append(exact.cpu())

                      total += labels.size(0)

                  return correct, total, all_halt_steps_validation, all_exact_validation, step_exact, step_count, step_tok_right, step_tok_total


            use_mask = True

            if use_mask:
              correct, total, all_halt_steps_validation, all_exact_validation, _, _, _, _ = validation(use_mask)
              print(f"\nValidation Accuracy with mask: {100 * correct / total:.2f}%")

              all_halt_steps_validation = torch.cat(all_halt_steps_validation)

              mean_h = all_halt_steps_validation.float().mean().item()
              median_h = all_halt_steps_validation.float().median().item()
              frac_1 = (all_halt_steps_validation == 1).float().mean().item()
              frac_max = (all_halt_steps_validation == n_sup).float().mean().item()

              dist = dict(Counter(all_halt_steps_validation.tolist()))

              print(f"validation halt steps: mean={mean_h:.2f} median={median_h:.0f} "
                    f"%step1={frac_1:.1%} %never={frac_max:.1%} dist={dist}")

              all_exact_validation = torch.cat(all_exact_validation)

              for s in range(1, n_sup + 1):
                  m = (all_halt_steps_validation == s)
                  if m.sum() > 0:
                      print(f"  halt_step_{s} | n={int(m.sum())} | EM: {100*all_exact_validation[m].float().mean().item():.2f}%")


          # Evaluate every depth with halting disabled.

            _, tot2, _, _, step_exact, step_count, step_tok_right, step_tok_total = validation(False)
            print("\nForced-depth (no halting):")
            print("        " + "".join(f"  chain={k}" for k in range(5)))

            for i in range(n_sup):
                row = "".join(f"  {100*step_exact[i,k].item()/max(step_count[i,k].item(),1):6.2f}%" for k in range(5))
                em  = 100 * step_exact[i].sum().item() / tot2
                tok = 100 * step_tok_right[i].item() / max(step_tok_total[i].item(), 1)
                print(f"  step_{i+1} |{row} | all EM: {em:.2f}% | TokAcc: {tok:.2f}%")

In [ ]:
# import torch
# import torch.nn.functional as F

# def model_training_and_validation(c, T, n_sup):
#     running_loss = 0
#     running_q = 0

#     # Use PAD_TOKEN for ignoring index in loss
#     criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

#     # Initialize model with correct sequence lengths
#     model = TinyRModel(hidden_size, vocab_size, inp_seq_len, out_seq_len).to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#     epochs = 100

#     print('Training Logs ------ \n')
#     for _ in range(epochs):
#         model.train()
#         for step, batch in enumerate(train_loader):
#             input_ids = batch['input_ids'].to(device)
#             labels = batch['labels'].to(device)

#             x = input_ids
#             # Initial states must be (Batch, hidden_size)
#             y = torch.zeros(input_ids.size(0), hidden_size, device=device)
#             z = torch.zeros(input_ids.size(0), hidden_size, device=device)

#             step_loss = step_q = 0
#             overall_loss = 0
#             n_taken = 0

#             for i in range(n_sup):
#                 n_taken = i + 1
#                 f_out, halt, y_new, z_new = model(x, y, z, T, n=6)
#                 y, z = y_new.detach(), z_new.detach()

#                 # Map f_out (B, out_seq_len * vocab_size) to (B * out_seq_len, vocab_size)
#                 seq_len = labels.size(1)

#                 logits = f_out.view(-1, vocab_size)
#                 targets = labels.view(-1)

#                 # print(halt.shape, targets.shape, labels.shape, logits.shape)

#                 loss = criterion(logits, targets)

#                 # print(f_out.shape, labels.shape)
#                 q_loss_val = Q_LOSS(halt, f_out.view(-1, seq_len, vocab_size), labels)

#                 overall_loss += (loss + c * q_loss_val)
#                 step_loss += loss.item()
#                 step_q += q_loss_val.item()

#                 if (halt > 0).all():
#                     break

#             overall_loss = overall_loss / n_taken
#             optimizer.zero_grad()
#             overall_loss.backward()
#             optimizer.step()

#             running_loss += step_loss / n_taken
#             running_q += step_q / n_taken

#             if step % 50 == 0 and step > 0 and n_taken<n_sup:
#                 print(f"Step {step} | CE: {running_loss/50:.4f} | Q-Loss: {running_q/50:.4f} | Avg Steps: {n_taken}")
#                 running_loss = running_q = 0

#         print('\nValidation Logs ------ \n')
#         model.eval()
#         correct = 0
#         total = 0
#         with torch.no_grad():
#             for val_step, batch in enumerate(val_loader):
#                 input_ids = batch['input_ids'].to(device)
#                 labels = batch['labels'].to(device)
#                 x = input_ids
#                 y = torch.zeros(input_ids.size(0), hidden_size, device=device)
#                 z = torch.zeros(input_ids.size(0), hidden_size, device=device)

#                 for i in range(n_sup):
#                     f_out_val, val_halt, y_new, z_new = model(x, y, z, T, n=6)
#                     y, z = y_new.detach(), z_new.detach()
#                     if (val_halt > 0).all(): break

#                 logits = f_out_val.view(-1, vocab_size)
#                 targets = labels.view(-1)
#                 predictions = logits.argmax(dim=1)

#                 mask = targets != PAD_TOKEN
#                 correct += (predictions[mask] == targets[mask]).sum().item()
#                 total += mask.sum().item()

#         if epoch % 10 == 0:
#             print(f"\nValidation Accuracy: {100 * correct / total:.2f}%")

In [ ]:
# Development check: these filters refer to the original small-operand task.
# The first two groups are empty for the fixed-length runs here.
easy_dataset = [d for d in dataset.data if d['num_digits'] <= 2 and d['max_carry_chain'] == 0]
print(f"Easy examples: {len(easy_dataset)}")

medium_dataset = [d for d in dataset.data if d['num_digits'] <= 2]
print(f"Medium examples: {len(medium_dataset)}")

# This group contains only 4-digit operands; it is empty in 8-digit runs.
hard_dataset = [d for d in dataset.data if d['num_digits'] == 4]
print(f"Hard examples: {len(hard_dataset)}")

Easy examples: 0
Medium examples: 0
Hard examples: 50000


In [ ]:
# 4-digit addition | seed=100 | target=softmean | epochs=100
# n_sup=5 | halt logit > 0 | c=0.1 | T=3

HALT_THRESHOLD = 0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=5, HALT_THRESHOLD = HALT_THRESHOLD)

PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6227 | Q-Loss: 0.6191 | Avg Steps: 5.0
halt logit: mean=-0.756 std=0.410 max=0.552 %>0=0.3% %>1.5=0.0%
halt steps: mean=4.96 median=5 %step1=0.8% %never=99.1% dist={5: 44578, 1: 356, 2: 45, 3: 14, 4: 7}

Validation Logs ------ 

sup_step_1 | mean=-0.673 std=0.386 median=-0.918 %>0=0.0% %>1.5=0.0% first5=[-1.0318272113800049, -0.23613348603248596, -0.9206455945968628, -1.1110939979553223, -0.9931870698928833]
sup_step_2 | mean=-0.718 std=0.339 median=-0.905 %>0=0.0% %>1.5=0.0% first5=[-1.0781692266464233, -0.39123600721359253, -0.9046392440795898, -1.1419225931167603, -1.0322338342666626]
sup_step_3 | mean=-0.716 std=0.347 median=-0.924 %>0=0.0% %>1.5=0.0% first5=[-1.0812116861343384, -0.36295148730278015, -0.9244862794876099, -1.1410505771636963, -1.0366679430007935]
sup_step_4 | mean=-0.715 std=0.349 median=-0.928 %>0=0.0% %>1.5=0.0% first5=[-1.0811716318130493, -0.3663196265697479, -0.927522420

In [ ]:
# 4-digit addition | seed=100 | target=softmean | epochs=100
# n_sup=5 | halt logit > 1.5 | c=0.1 | T=3

HALT_THRESHOLD = 1.5
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=5, HALT_THRESHOLD = HALT_THRESHOLD)

PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6221 | Q-Loss: 0.6199 | Avg Steps: 5.0
halt logit: mean=-0.756 std=0.406 max=0.559 %>0=0.3% %>1.5=0.0%
halt steps: mean=5.00 median=5 %step1=0.0% %never=100.0% dist={5: 45000}

Validation Logs ------ 

sup_step_1 | mean=-0.685 std=0.381 median=-0.906 %>0=0.0% %>1.5=0.0% first5=[-1.0177850723266602, -0.2402482032775879, -0.9664636850357056, -1.1186500787734985, -1.0030032396316528]
sup_step_2 | mean=-0.746 std=0.326 median=-0.900 %>0=0.0% %>1.5=0.0% first5=[-1.0678400993347168, -0.4400365650653839, -0.953549861907959, -1.1516525745391846, -1.0410468578338623]
sup_step_3 | mean=-0.740 std=0.338 median=-0.924 %>0=0.0% %>1.5=0.0% first5=[-1.0701302289962769, -0.3996773660182953, -0.971894383430481, -1.1566874980926514, -1.044108271598816]
sup_step_4 | mean=-0.741 std=0.337 median=-0.926 %>0=0.0% %>1.5=0.0% first5=[-1.0700680017471313, -0.4053370952606201, -0.9745094776153564, -1.1566648483276367, -1.

In [ ]:
# 4-digit addition | seed=100 | target=softmean | epochs=100
# n_sup=1 | halt logit > 0 | c=0.1 | T=3

HALT_THRESHOLD = 0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=1, HALT_THRESHOLD = HALT_THRESHOLD)

PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6149 | Q-Loss: 0.6229 | Avg Steps: 1.0
halt logit: mean=-0.736 std=0.408 max=0.585 %>0=0.3% %>1.5=0.0%
halt steps: mean=1.00 median=1 %step1=100.0% %never=100.0% dist={1: 45000}

Validation Logs ------ 

sup_step_1 | mean=-0.694 std=0.311 median=-0.837 %>0=0.0% %>1.5=0.0% first5=[-1.0551677942276, -0.2555566430091858, -0.9028186798095703, -1.0003647804260254, -1.0470960140228271]

Validation Accuracy with mask: 0.04%
validation halt steps: mean=1.00 median=1 %step1=100.0% %never=100.0% dist={1: 5000}
  halt_step_1 | n=5000 | EM: 0.04%

Forced-depth (no halting):
          chain=0  chain=1  chain=2  chain=3  chain=4
  step_1 |    0.00%    0.10%    0.00%    0.00%    0.00% | all EM: 0.04% | TokAcc: 36.43%
Training processed epoch 1
Training processed epoch 2
Training processed epoch 3
Training processed epoch 4
Training processed epoch 5
Training processed epoch 6
Training processed epoch 7
Training

In [ ]:
# 4-digit addition | seed=100 | target=softmean | epochs=100
# n_sup=5 | halt logit > 3.0 | c=0.1 | T=3

HALT_THRESHOLD = 3.0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=5, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6221 | Q-Loss: 0.6199 | Avg Steps: 5.0
halt logit: mean=-0.756 std=0.406 max=0.559 %>0=0.3% %>1.5=0.0%
halt steps: mean=5.00 median=5 %step1=0.0% %never=100.0% dist={5: 45000}

Validation Logs ------ 

sup_step_1 | mean=-0.685 std=0.381 median=-0.906 %>0=0.0% %>1.5=0.0% first5=[-1.0177850723266602, -0.2402482032775879, -0.9664636850357056, -1.1186500787734985, -1.0030032396316528]
sup_step_2 | mean=-0.746 std=0.326 median=-0.900 %>0=0.0% %>1.5=0.0% first5=[-1.0678400993347168, -0.4400365650653839, -0.953549861907959, -1.1516525745391846, -1.0410468578338623]
sup_step_3 | mean=-0.740 std=0.338 median=-0.924 %>0=0.0% %>1.5=0.0% first5=[-1.0701302289962769, -0.3996773660182953, -0.971894383430481, -1.1566874980926514, -1.044108271598816]
sup_step_4 | mean=-0.741 std=0.337 median=-0.926 %>0=0.0% %>1.5=0.0% first5=[-1.0700680017471313, -0.4053370952606201, -0.9745094776153564, -1.1566648483276367, -1.

In [ ]:
# 4-digit addition | seed=100 | target=softmean | epochs=100
# n_sup=16 | halt logit > 0 | c=0.1 | T=3

HALT_THRESHOLD = 0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=16, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6257 | Q-Loss: 0.6183 | Avg Steps: 16.0
halt logit: mean=-0.760 std=0.408 max=0.407 %>0=0.3% %>1.5=0.0%
halt steps: mean=15.75 median=16 %step1=1.5% %never=98.3% dist={16: 44231, 1: 684, 2: 65, 3: 15, 4: 4, 5: 1}

Validation Logs ------ 

sup_step_1 | mean=-0.693 std=0.385 median=-0.860 %>0=3.1% %>1.5=0.0% first5=[-1.1101770401000977, -0.40071263909339905, -0.9844306111335754, -1.1136120557785034, -1.0971177816390991]
sup_step_2 | mean=-0.694 std=0.319 median=-0.772 %>0=0.0% %>1.5=0.0% first5=[-1.0454474687576294, -0.3970397114753723, -0.9255090951919556, -1.0964597463607788, -1.0260378122329712]
sup_step_3 | mean=-0.705 std=0.319 median=-0.816 %>0=0.0% %>1.5=0.0% first5=[-1.0481897592544556, -0.44176575541496277, -0.9538357853889465, -1.0979686975479126, -1.0310004949569702]
sup_step_4 | mean=-0.695 std=0.331 median=-0.813 %>0=0.0% %>1.5=0.0% first5=[-1.0473365783691406, -0.42189520597457886, -0

## Saved output

Each of the five main runs has 100 processed epochs; detailed metrics appear every ten epochs. The separate diagnostic run has 100 epochs. Its output and the dataset previews are also saved in this notebook.


`c=0.1` throughout the five main runs. No loss-weight ablation was run.
